# GPT2 Medium Experiments

## Introduction

I state that the current model scale of my experimentation is too small, that given I am only tested the redundancy across 4 heads that this scale of experiment is too small to fully quantify if my idea of applying a MLA inspired compress/decompress bottleneck to the output projection achieves a reduction in the number of model parameters while retaining the models quality. 

To avoid a costly large scale runs, I wish to establish if this might work before the fact, by leveraging already trained models and replacing their output projection matrices with lower rank matrices of different levels of reduction, and then evaluating these models on an evaluation dataset. I will be evaluating the GPT2 small, medium and large models which have 12, 16 and 20 attention heads respectively per model layer, much larger than the 4 that I have been evaluating in my experiment.

Therefore, in this notebook I will first take a trained model, GPT2 Small, Medium and Large, and replace their respective output projection matrix with a lower rank matrix of size $r$, where $r$ represents the fraction of full rank of the matrix. Perplexity will be evaluated on the wikitext-103-raw-v1 evaluation dataset after the replacement. The idea being if I can show that the perplexity curve against $r$, the fraction of full rank, show a similar or different pattern across the three models. Ideally, the curves will remain flat for a large portion of the rank percentages and especially we want to see the larger models stay flat for longer. This would confirm that my experimentation may work and would be worth exploring further.

Each model will also be finetuned on the training data from wikitext-103-raw-v1 for a fix small number of steps approx 200. The idea being that the model will adapt to this new dataset with the new architecture.

## Define the necessary libraries

In [2]:
# Base libraries
import random

# Maths
import math
import os
import json
import hashlib
import numpy as np
import torch
import pandas as pd
from torch import nn
from tqdm.notebook import tqdm
from pathlib import Path
import hydra
from omegaconf import DictConfig
from dataclasses import dataclass

import gc
from accelerate.state import AcceleratorState, PartialState

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers.utils import logging
from datasets import load_from_disk
from torch.utils.data import DataLoader
from torch.optim import AdamW

# MLA
from mla.config.paths import get_pretraining_paths, get_finetuning_paths
from mla.model_training.model_training import ModelPreTraining
from mla.model_training.strategies import _PRETRAINING_STRATEGIES, _FINETUNING_STRATEGIES
from mla.model_training.model_finetuning import model_finetuning
from mla.utils.data_preparation import import_and_prepare_data, prepare_dataloaders
from mla.utils.setup import set_all_seeds
from mla.utils.progress import ProgressBars
from mla.config.paths import Paths, build_dataset_name, TRAINING_DATA_DIR, TRAINING_MODELS_DIR

In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [4]:
# Define env variables
os.environ["WANDB_SILENT"] = "true"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [5]:
# Disable transformer logging 
logging.disable_progress_bar()

## General Configuration

In [6]:
# Define the config
config = {
    "seed": 42,
    "dataset_name": "Salesforce/wikitext",
    "dataset_config_name": "wikitext-103-raw-v1",
    "max_load_pct": None,
    "train_token_budget": None, 
    "val_token_budget": None,
    "sentence_keys": ["text"],
    "mlm": False,
    "mlm_probability": None,
    "root_dir": "data",
    
    "batch_size": 1,
    "drop_last": True,
    "eval_batch_size": 4,
    "gradient_accumulation_steps": 16,
    "mixed_precision": "fp16",
    "weight_decay": 0.01,
    "learning_rate": 5e-4,
    "keep_last_n_checkpoints": None,
    "experiment_name": None,
    "job_type": None,
    "eval_steps": 500,
    "eval_metric": "loss",
    "loss_type": "ForCausalLM",
    
    "task_type": "pretraining",
    "parallel_processes": 8,
    "num_workers": 0,
    "pin_memory": False,
    "hidden_size": None,
    "n_layer": None,
    "n_head": None,
    "max_seq_length": 1024, 
    "attention_mechanism": "MHA",
    "kv_compression_dim": None,
    "q_compression_dim": None,
    "output_compression_dim": None,
    "experiment_project": "GPT2_Testing",
    "r_thresholds": [1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125],
    
    "num_labels": 0,
    "max_steps": 200,
    "warmup": True,
    "warmup_rate": 0.05,
    "max_norm": 1.0,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "train_eval_steps": 500,
    "eval_steps": 500,
}

## General Functions

### Apply SVD to each $W_O$ matrix

In [7]:
class OutputProjection(torch.nn.Module):
    def __init__(self, A, B, bias):
        super().__init__()
        self.A = torch.nn.parameter.Parameter(A.detach())
        self.B = torch.nn.parameter.Parameter(B.detach())
        self.bias = torch.nn.parameter.Parameter(bias.detach())

    def forward(self, x):
        return ((x @ self.A) @ self.B) + self.bias

In [8]:
def keep_top_r(U, S, VT, r_threshold):
    """
    Retain the top r percentage of the full rank.
    """
    full_rank = len(S)
    rank_fraction = int(full_rank * r_threshold)
    s_sqrt = torch.sqrt(S[:rank_fraction])
    A = (U[:, :rank_fraction] * s_sqrt)
    B = (s_sqrt.unsqueeze(1) * VT[:rank_fraction])
    return A, B
    

In [9]:
def calculate_SVD_rank_metrics(WO, r_threshold):
    """
    First calculate the SVD and then calculate the retension based rank value and entropy baesd effective rank values. 
    """
    
    # Calculate the SVD
    U, S, VT = torch.linalg.svd(WO, full_matrices=False)

    # Calculate the retension based rank values
    A, B = keep_top_r(U, S, VT, r_threshold)

    return A, B

### Evaluate the model before finetuning

In [10]:
def eval_model_perplexity(model, validation_loader, root_dir, config):
    """
    Evaluate the models perplexity.
    """

    # Define the model pretraining paths
    paths = get_pretraining_paths(root_dir, config, config.seed)

    # Define the pretraining object
    pretraining = ModelPreTraining(
        model=model,
        optimizer=AdamW,
        metric_fn=_PRETRAINING_STRATEGIES[config.base_model].metric_cls(),
        config=config,
        paths=paths,
        seed=config.seed,
    )
    
    # Run the evaluation
    pretraining_results = pretraining.eval_model(validation_loader, training_eval=False)
    
    # Calculate the perplexity of the model
    model_loss = pretraining_results[0].avg
    model_perplexity = math.exp(model_loss)
    
    return model_loss, model_perplexity


### Model Finetuning (via pretraining path) + evaluate

In [11]:
def finetune_model(model, train_loader, validation_loader, root_dir, config):
    """
    Finetune the model for 200 steps on the Wikitext training dataset.
    """

    # Define the model pretraining paths
    paths = get_pretraining_paths(root_dir, config, config.seed)

    # Define the pretraining object
    pretraining = ModelPreTraining(
        model=model,
        optimizer=AdamW,
        metric_fn=_PRETRAINING_STRATEGIES[config.base_model].metric_cls(),
        config=config,
        paths=paths,
        seed=config.seed,
    )

    # Finetune the model for 200 steps on the training dataset
    pretraining.model_training(training_dataloader=train_loader, validation_dataloader=validation_loader)

    # Evaluate the model again after finetuning
    finetuning_results = pretraining.eval_model(validation_loader, training_eval=False)

    # Calculate the perplexity of the model
    ft_loss = finetuning_results[0].avg
    ft_perplexity = math.exp(ft_loss)

    return ft_loss, ft_perplexity

### Evaluate the model before and after of finetuning

In [12]:
def evaluate_gpt2_model(config):
    """
    Evaluate the GPT2 model before and after finetuning on 200 steps of WikiText V2 dataset.
    """
    # Define the results dictionary
    results_dict = {"model": [], "r": [], "actual_loss": [], "actual_ppl": [], "finetuning_loss": [], "finetuning_ppl": []}
    root_dir = Path(config.root_dir)
    
    # For each threshold
    for r_threshold in tqdm(config.r_thresholds):
    
        # Import the model
        model = GPT2LMHeadModel.from_pretrained(config.model_name, local_files_only=False)
    
        # Extract the WO weight matrices
        for i, layer in enumerate(model.transformer.h):
        
            # Identify the weight matrix
            bias = layer.attn.c_proj.bias
            WO = layer.attn.c_proj.weight
            
            # Calculate the entropy based effective rank and also retention based rank values
            A, B = calculate_SVD_rank_metrics(WO, r_threshold=r_threshold)
        
            # Update the model weights
            layer.attn.c_proj = OutputProjection(A, B, bias)
    
        # Evaluate the perplexity of the model
        actual_loss, actual_ppl = eval_model_perplexity(model, validation_loader, root_dir, config)
    
        # Finetune the model on a set of training data
        finetuning_loss, finetuning_ppl = finetune_model(model, train_loader, validation_loader, root_dir, config)
    
        # Store the results
        results_dict["model"].append(config.model_name)
        results_dict["r"].append(r_threshold)
        results_dict["actual_loss"].append(actual_loss)
        results_dict["actual_ppl"].append(actual_ppl)
        results_dict["finetuning_loss"].append(finetuning_loss)
        results_dict["finetuning_ppl"].append(finetuning_ppl)

    return pd.DataFrame(results_dict)

## Load the WikiText datasets

In [13]:
def load_dataset(tokenizer, config):
    """
    Load the WikiText dataset and prepare dataloaders for the training and validation datasets.
    """
    # Ensure that it exists
    root_dir = Path(config.root_dir)
    root_dir.mkdir(parents=True, exist_ok=True)
    
    # Define the dataset paths
    data_paths = get_pretraining_paths(root_dir, config)
    
    # Prepare the datasets
    datasets = import_and_prepare_data(tokenizer, config, data_paths.tokenized_data_path)
    
    # Prepare the training dataloaders for finetuning
    train_loader, validation_loader = prepare_dataloaders(datasets, tokenizer, config, collator_fn=None)

    return train_loader, validation_loader

## GPT2 Medium

In [14]:
# Wrap the config in a DictConfig
config = DictConfig(config)
config["base_model"] = "GPT2"
config["model_name"] = "gpt2-medium"

In [15]:
# Set all seeds
set_all_seeds(config.seed)

In [16]:
# Define the TinyGPT2 model
tokenizer = GPT2Tokenizer.from_pretrained(config.model_name, local_files_only=False)

In [17]:
# Load the training and validation datasets
train_loader, validation_loader = load_dataset(tokenizer, config)

Loading Salesforce/wikitext/wikitext-103-raw-v1 dataset from: data/training_data/GPT2_Testing/pretraining/Salesforce__wikitext_wikitext-103-raw-v1_2b2a1bd31f



In [18]:
# Generate a set of results
gpt2_medium_results = evaluate_gpt2_model(config)

  0%|          | 0/1 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Validation:   0%|                                                                 | 0/59 [10:03<?, ?it/s]


r=0.03125  allocated=0.02 GB


## Save the results

In [19]:
def save_results(df: pd.DataFrame, path: str = "data/svd_probe_results.csv") -> None:
    """Append a results DataFrame to a shared CSV (writes header only on first write)."""
    path = Path(path)
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

In [20]:
# Save the results locally
save_results(gpt2_medium_results)

## Conclusion

Due to CUDA memory constraints these experiments had to be run individually. 